# 01 - Reproduce the IDIAP text teacher

Reuses a strictly validated persistent cache of the **exact external text-teacher KD targets** when available; otherwise it builds that cache once from the pinned public IDIAP checkpoint. This is frozen-checkpoint inference, not random re-training, because that is how the v3 teacher was defined.

**Selection policy:** teacher epoch 0 is the original published IDIAP checkpoint. Any future local fine-tuning must beat epoch 0 under the fixed DEV criterion before it may replace the original teacher.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, shutil, subprocess, base64
import pandas as pd

DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
EXP_ROOT = DAIC / 'experiments' / 'exact_v3_reproduction'
ACTIVE = EXP_ROOT / 'ACTIVE_RUN.txt'
assert ACTIVE.is_file(), 'Run notebooks/00_start_new_run.ipynb first.'
RUN = EXP_ROOT / ACTIVE.read_text().strip()
assert RUN.is_dir(), RUN

REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
CORE_COMMIT = '0d6ac621be7da1db7c1d769d075d142d0a740fd8'
CODE = Path('/content/reliability-aware-depression-kd-exact-v3')

token = userdata.get('GITHUB_TOKEN')
assert token, 'Add GITHUB_TOKEN to Colab Secrets.'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy()
env['GIT_CONFIG_COUNT']='1'
env['GIT_CONFIG_KEY_0']='http.https://github.com/.extraheader'
env['GIT_CONFIG_VALUE_0']=f'AUTHORIZATION: basic {auth}'

if CODE.exists(): shutil.rmtree(CODE)
CODE.mkdir(parents=True)
subprocess.run(['git','-C',str(CODE),'init'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'remote','add','origin',f'https://github.com/{REPO}.git'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',CORE_COMMIT],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'checkout','--detach','FETCH_HEAD'],check=True,env=env)
head=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
assert head==CORE_COMMIT,(head,CORE_COMMIT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
print('RUN:',RUN)
print('Pinned core:',head)


In [ ]:
IDIAP_REPO='https://github.com/idiap/bias_in_daic-woz.git'
IDIAP_COMMIT='de8f477619ce61a2a65640a0735aa8dfe659ac59'
SRC=Path('/content/idiap-bias_in_daic-woz')
if SRC.exists(): shutil.rmtree(SRC)
subprocess.run(['git','clone',IDIAP_REPO,str(SRC)],check=True)
subprocess.run(['git','-C',str(SRC),'checkout','--detach',IDIAP_COMMIT],check=True)
resolved=subprocess.check_output(['git','-C',str(SRC),'rev-parse','HEAD'],text=True).strip()
assert resolved==IDIAP_COMMIT,(resolved,IDIAP_COMMIT)
print('Pinned IDIAP source:',resolved)

In [ ]:
OUT=RUN/'01_text_teacher'
OUT.mkdir(parents=True,exist_ok=False)

# Persistent deterministic teacher-target cache.
CANONICAL_TEXT_CACHE=DAIC/'experiments'/'idiap_text_teacher'
SHARED_TEXT_CACHE=EXP_ROOT/'_shared_cache'/'idiap_text_teacher_epoch0_v1'
TEXT_CACHE_POINTER=RUN/'01_text_teacher_cache.json'

def valid_text_teacher_cache(root):
    root=Path(root)
    need=[
        root/'train_text_kd_targets.csv',
        root/'dev_text_predictions.csv',
        root/'text_target_audit.json',
    ]
    if not all(p.is_file() for p in need):
        return False,None
    try:
        audit=json.loads((root/'text_target_audit.json').read_text())
        tr=pd.read_csv(root/'train_text_kd_targets.csv')
        dv=pd.read_csv(root/'dev_text_predictions.csv')
    except Exception:
        return False,None

    m=audit.get('dev34',{})
    ok=(
        audit.get('train_participants')==107 and
        audit.get('dev_participants')==34 and
        audit.get('test_opened') is False and
        len(tr)==107 and tr['participant_id'].nunique()==107 and
        len(dv)==34 and dv['participant_id'].nunique()==34 and
        440 not in set(dv['participant_id'].astype(int)) and
        round(float(m.get('macro_f1',-1)),10)==round(0.8418604651162791,10)
    )
    info={
        'path':str(root),
        'train_participants':int(audit.get('train_participants',-1)),
        'dev_participants':int(audit.get('dev_participants',-1)),
        'participant_440_excluded':440 not in set(dv['participant_id'].astype(int)) if len(dv) else False,
        'dev_macro_f1':float(m.get('macro_f1',float('nan'))),
        'test_opened':audit.get('test_opened'),
    }
    return ok,info

CACHE=None; cache_info=None; cache_source=None
for candidate,label in [
    (CANONICAL_TEXT_CACHE,'existing canonical idiap_text_teacher cache'),
    (SHARED_TEXT_CACHE,'exact-v3 shared text-teacher cache'),
]:
    ok,info=valid_text_teacher_cache(candidate)
    if ok:
        CACHE=Path(candidate); cache_info=info; cache_source=label; break

if CACHE is None:
    CACHE=SHARED_TEXT_CACHE
    CACHE.mkdir(parents=True,exist_ok=True)
    cmd=[sys.executable,'-m','scripts.teachers.idiap_text.export_targets',
         '--daic-root',str(DAIC),'--source-root',str(SRC),'--output',str(CACHE)]
    print('No valid text-teacher cache found. Building ONCE at:',CACHE)
    subprocess.run(cmd,cwd=CODE,check=True)
    ok,cache_info=valid_text_teacher_cache(CACHE)
    assert ok,'New IDIAP text-teacher cache failed validation.'
    cache_source='new exact-v3 shared text-teacher cache'

# Copy only compact deterministic results into the new run; no recomputation.
for name in ['train_text_kd_targets.csv','dev_text_predictions.csv','text_target_audit.json']:
    shutil.copy2(CACHE/name,OUT/name)

TEXT_CACHE_POINTER.write_text(json.dumps({
    'cache_type':'IDIAP epoch-0 teacher targets/predictions',
    'cache_source':cache_source,
    **cache_info,
    'immutable_for_this_run':True,
    'run_id':RUN.name,
},indent=2)+'\n')

audit=json.loads((OUT/'text_target_audit.json').read_text())
m=audit['dev34']
print('REUSING TEXT-TEACHER CACHE:',CACHE)
print(json.dumps(m,indent=2))
assert audit['train_participants']==107 and audit['dev_participants']==34
assert audit['test_opened'] is False
assert round(float(m['macro_f1']),10)==round(0.8418604651162791,10)

# Teacher epoch 0 means the ORIGINAL published checkpoint.
orig_model=SRC/'model/Participant/model_inductgcn[250].pkl'
orig_vectorizer=SRC/'model/Participant/vtzer_inductgcn[250].pkl'
assert orig_model.is_file() and orig_vectorizer.is_file()

shutil.copy2(orig_model,OUT/'epoch0_original_teacher.pkl')
shutil.copy2(orig_vectorizer,OUT/'epoch0_original_vectorizer.pkl')
shutil.copy2(orig_model,OUT/'selected_teacher.pkl')
shutil.copy2(orig_vectorizer,OUT/'selected_vectorizer.pkl')
shutil.copy2(OUT/'train_text_kd_targets.csv',OUT/'selected_train_kd_targets.csv')

selection={
    'teacher':'IDIAP participant InducT-GCN top250',
    'epoch0_definition':'original published checkpoint before any local fine-tuning',
    'epoch0_source_commit':IDIAP_COMMIT,
    'epoch0_dev34':m,
    'selection_rule':'start with epoch 0; replace only on strict lexicographic DEV improvement: macro-F1, depressed-F1, AUROC',
    'fine_tuning_attempted':False,
    'selected_epoch':0,
    'selected_source':'original published checkpoint',
    'selected_targets':'selected_train_kd_targets.csv',
    'teacher_target_cache':str(CACHE),
    'test_opened':False,
}
(OUT/'teacher_selection.json').write_text(json.dumps(selection,indent=2)+'\n')

print('PASS: IDIAP text teacher DEV-34 macro-F1 =',m['macro_f1'])
print('SELECTED TEACHER: epoch 0 original published IDIAP checkpoint')
print('No text-teacher target recalculation was needed.')